# 🚀 MyBabyAI - Turbo Training Notebook

Optimized for high-end hardware (90GB VRAM+, A100/H100/RTX4090).

This notebook enables advanced features like **Flash Attention 2**, **Torch Compile**, **BF16**, and a **4096 sequence length** to maximize training speed and quality for the `CodeMind-650M` model.

In [ ]:
# @title 1. 🛠️ Environment Setup & Optimizations
import os
import sys
import torch
import gc
from pathlib import Path

# --- ⚡ PRE-IMPORT OPTIMIZATIONS ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def setup_turbo_env():
    print("🚀 Initializing Turbo Training Environment...")
    if not torch.cuda.is_available():
        print("❌ CUDA is not available. This script requires a high-end GPU.")
        return
    
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Detected: {gpu_name} | VRAM: {total_mem:.2f} GB")
    
    # Enable TF32 for speedup on Ampere+ GPUs
    if torch.cuda.get_device_capability()[0] >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        print("⚡ TF32 Enabled")

setup_turbo_env()

# Project Root Setup
project_root = Path(".").absolute()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"📂 Project Root: {project_root}")

In [ ]:
# @title 2. 📝 Loading Turbo Configuration
from src.utils.config import Config
from src.core.model_manager import ModelManager
from src.core.trainer import LoRATrainer

turbo_config_path = "configs/turbo_config.yaml"
if not os.path.exists(turbo_config_path):
    print(f"❌ Turbo config file not found at {turbo_config_path}. Falling back to default or creating internal config.")
    # You can manually set config here if needed
    config = Config() 
else:
    config = Config(config_path=turbo_config_path)
    print(f"✅ Loaded Turbo Configuration: {turbo_config_path}")

In [ ]:
# @title 3. 🌱 Initializing Model
model_manager = ModelManager(config)
model_size = "650M"
training_mode = "full" # Full training for 90GB VRAM

print(f"🚀 Loading CodeMind-{model_size}...")
model_manager.load_fresh_model(size=model_size)

# Clear VRAM
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# @title 4. 📊 Dataset Pool Setup
dataset_pool = [
    {
        "name": "🇹🇷 Turkish Instructions (Merve)",
        "type": "huggingface",
        "dataset_key": "turkish_instructions_merve",
        "max_samples": 5000
    },
    {
        "name": "🇹🇷 Turkish Alpaca",
        "type": "huggingface",
        "dataset_key": "turkish_alpaca",
        "max_samples": 5000
    },
    {
        "name": "🇹🇷 GPT-4 Alpaca TR",
        "type": "huggingface",
        "dataset_key": "alpaca_gpt4_tr",
        "max_samples": 5000
    },
    {
        "name": "🇹🇷 InstrucTurca",
        "type": "huggingface",
        "dataset_key": "instructurca",
        "lazy_load": True,
        "max_samples": 50000 
    }
]

print(f"✅ Dataset pool configured with {len(dataset_pool)} sources.")

In [ ]:
# @title 5. ⚡ Start Turbo Training
trainer = LoRATrainer(model_manager, config)

print("🚀 Starting Turbo Training Run...")
try:
    trainer.train_from_pool(
        dataset_pool,
        training_type=training_mode,
        max_length=config.get("training.max_length", 4096),
        use_notebook_callback=True # Enabled for rich progress bars in Notebook
    )
    print("\n✅ Training completed successfully!")
except Exception as e:
    print(f"\n❌ Training failed: {e}")